In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys

import torch

sys.path.insert(0, "../../src")

from juart.vis.interactive import InteractiveMultiPlotter3D, MAE

from juart.conopt.functional.fourier import (
    fourier_transform_adjoint,
    fourier_transform_forward,
    nonuniform_fourier_transform_adjoint,
)

In [ ]:
device = "cpu"
normalize_images = True

images = []

images.append(torch.load("../../../images/num_cgsense_reco/meas_preproc-2_R1/120i").to(device).cpu().abs().numpy())
images.append(torch.load("../../../images/num_cgsense_reco/meas_preproc-2_R10/120i").to(device).cpu().abs().numpy())

for i in range(6,32,5):
    images.append(torch.load(f"../../../images/model_test_k100_i0/model_epoch_{i}").to(device).cpu().abs().numpy()[:,:,:,0,0])
# images.append(torch.load(f"../../../images/fibo_UNet_01i_50DC_50_f256_R4/model_epoch_4").to(device).cpu().abs().numpy()[:,:,:,0,0])
# images.append(torch.load(f"../../../images/fibo_UNet_01i_50DC_50_f256_R4/model_epoch_6").to(device).cpu().abs().numpy()[:,:,:,0,0])

In [ ]:
if normalize_images:
    for image in images:
        image /= image[20:100,20:100,65].max()

In [ ]:
InteractiveMultiPlotter3D(images,
                          layout = [2,4],
                          title = ["cgSENSE_R1", "cgSENSE_R10","UNet E4","E7","E10","E13","E16","E19"],
                          cmap="gray",
                          vmin = 0,
                          vmax = 1,
                          compact_plotting = True,
                          show_axis = False,
                          activate_colorbar = False,
                          reference = 0).interactive

In [ ]:
FT_images = []

for image in images:
    FT_images.append(fourier_transform_forward(torch.from_numpy(image),(0,1,2)).abs().numpy())

In [ ]:
InteractiveMultiPlotter3D(FT_images,
                          layout = [2,4],
                          title = ["cgSENSE_R1", "cgSENSE_R10","UNet E4","E7","E10","E13","E16","E19"],
                          cmap="gray",
                          vmin = 0,
                          vmax = 1,
                          compact_plotting = True,
                          show_axis = False,
                          activate_colorbar = False,
                          reference = 0).interactive

In [ ]:
def lowpass_filter(kspace_image, radius):
    FT = torch.from_numpy(kspace_image)
    coords = torch.stack(torch.meshgrid(
        torch.arange(FT.shape[0]),
        torch.arange(FT.shape[0]),
        torch.arange(FT.shape[0]),
        indexing='ij'
    ), dim=-1)

    center = (FT.shape[0]) / 2.0
    dist = torch.sqrt(torch.sum((coords - center)**2, dim=-1))

    mask = (dist <= radius).float()

    FT = FT * mask

    return FT.numpy()

In [ ]:
filtered_kspace = []

for FT in FT_images:
    filtered_kspace.append(lowpass_filter(FT,64))

In [ ]:
InteractiveMultiPlotter3D(filtered_kspace,
                          layout = [2,4],
                          title = ["cgSENSE_R1", "cgSENSE_R10","UNet E4","E7","E10","E13","E16","E19"],
                          cmap="gray",
                          vmin = 0,
                          vmax = 1,
                          compact_plotting = True,
                          show_axis = False,
                          activate_colorbar = False,
                          reference = 0).interactive

In [ ]:
filtered_images = []

for image in filtered_kspace:
    filtered_images.append(fourier_transform_adjoint(torch.from_numpy(image),(0,1,2)).abs().numpy())

In [ ]:
InteractiveMultiPlotter3D(filtered_images,
                          layout = [2,4],
                          title = ["cgSENSE_R1", "cgSENSE_R10","UNet E4","E7","E10","E13","E16","E19"],
                          cmap="gray",
                          vmin = 0,
                          vmax = 1,
                          compact_plotting = True,
                          show_axis = False,
                          activate_colorbar = False,
                          reference = 0).interactive